In [5]:
import cv2
import os
import matplotlib.pyplot as plt
import numpy as np

In [6]:
import tensorflow as tf
model = tf.keras.models.load_model("model.keras")

# if eyes are closed for unusual time, like more than blinks (taken as 2 sec), alarm generated

In [8]:
import cv2
import numpy as np
import time
import winsound

# Alarm settings
frequency = 2500
duration = 1000

# Load cascades
faceCascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_eye.xml")

# Load your trained model
# from tensorflow.keras.models import load_model
# model = load_model("your_model.h5")

# Webcam
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise IOError("Cannot open webcam")

start_time = None

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    eye_rois = []  # store both eyes

    # Detect faces
    faces = faceCascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        face_gray = gray[y:y+h, x:x+w]
        face_color = frame[y:y+h, x:x+w]

        # Detect eyes inside face
        eyes = eye_cascade.detectMultiScale(face_gray, 1.1, 4)

        # limit to 2 eyes
        eyes = eyes[:2]

        for (ex, ey, ew, eh) in eyes:
            eye = face_color[ey:ey+eh, ex:ex+ew]
            eye_rois.append(eye)

            # draw BOTH eyes
            cv2.rectangle(face_color, (ex, ey), (ex+ew, ey+eh), (0,255,0), 2)

        # draw face
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255,0,0), 2)

    font = cv2.FONT_HERSHEY_SIMPLEX

    # Prediction using BOTH eyes
    if len(eye_rois) > 0:
        preds = []

        for eye in eye_rois:
            img = cv2.resize(eye, (224, 224))
            img = np.expand_dims(img, axis=0)
            img = img / 255.0

            pred = model.predict(img, verbose=0)[0][0]
            preds.append(pred)

        # average prediction
        final_pred = sum(preds) / len(preds)

        if final_pred > 0.5:
            status = "Open Eyes"
            start_time = None

            cv2.putText(frame, status, (100, 100),
                        font, 1.5, (0,255,0), 2)

        else:
            status = "Closed Eyes"

            cv2.putText(frame, status, (100, 100),
                        font, 1.5, (0,0,255), 2)

            if start_time is None:
                start_time = time.time()

            elif time.time() - start_time > 2:
                cv2.putText(frame, "SLEEP ALERT!", (100, 200),
                            font, 1.5, (0,0,255), 3)

                winsound.Beep(frequency, duration)

    else:
        cv2.putText(frame, "No Eyes Detected",
                    (100, 100), font, 1.2, (255,255,0), 2)

    cv2.imshow("Drowsiness Detection", frame)

    if cv2.waitKey(2) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()